# 🛒 Análisis Alura Store Latam
### Desafío: ¿Qué tienda debe vender el Sr. Juan?
---


### 📥 Importación de datos


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

url  = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_1%20.csv"
url2 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_2.csv"
url3 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_3.csv"
url4 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_4.csv"

tienda1 = pd.read_csv(url)
tienda2 = pd.read_csv(url2)
tienda3 = pd.read_csv(url3)
tienda4 = pd.read_csv(url4)

# Agregar columna identificadora
tienda1['Tienda'] = 'Tienda 1'
tienda2['Tienda'] = 'Tienda 2'
tienda3['Tienda'] = 'Tienda 3'
tienda4['Tienda'] = 'Tienda 4'

# DataFrame unificado
df = pd.concat([tienda1, tienda2, tienda3, tienda4], ignore_index=True)

print('Datos cargados correctamente.')
print(f'Total de registros: {len(df)}')
df.head()

# 1. 💰 Análisis de Facturación


In [ ]:
# Calcular ingresos totales por tienda (Precio + Costo de envío)
facturacion = df.groupby('Tienda')[['Precio', 'Costo de envío']].sum()
facturacion['Ingreso Total'] = facturacion['Precio'] + facturacion['Costo de envío']
facturacion = facturacion.sort_values('Ingreso Total', ascending=False)

print('=== Facturación por Tienda ===')
for tienda, row in facturacion.iterrows():
    print(f"{tienda}: ${row['Ingreso Total']:,.0f} COP")

print(f"\nTienda con MAYOR facturación: {facturacion['Ingreso Total'].idxmax()}")
print(f"Tienda con MENOR facturación: {facturacion['Ingreso Total'].idxmin()}")

In [ ]:
# Gráfico de barras - Facturación por tienda
colores = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(facturacion.index, facturacion['Ingreso Total'] / 1e9,
              color=colores, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, facturacion['Ingreso Total']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'${val/1e9:.2f}B', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Facturación Total por Tienda (COP)', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Ingresos (Miles de Millones COP)', fontsize=11)
ax.set_xlabel('Tienda', fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# 2. 📦 Ventas por Categoría


In [ ]:
# Ventas por categoría por tienda
cat_tienda = df.groupby(['Tienda', 'Categoría del Producto']).size().unstack(fill_value=0)

print('=== Número de ventas por Categoría y Tienda ===')
print(cat_tienda.to_string())

In [ ]:
# Gráfico de barras apiladas - Ventas por categoría
fig, ax = plt.subplots(figsize=(11, 6))

cat_tienda.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', linewidth=0.5)

ax.set_title('Ventas por Categoría en cada Tienda', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Número de Ventas', fontsize=11)
ax.set_xlabel('Tienda', fontsize=11)
ax.legend(title='Categoría', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 3. ⭐ Calificación Promedio de la Tienda


In [ ]:
# Calificación promedio por tienda
calificacion = df.groupby('Tienda')['Calificación'].mean().round(2).sort_values(ascending=False)

print('=== Calificación Promedio por Tienda ===')
for tienda, cal in calificacion.items():
    estrellas = '★' * int(round(cal)) + '☆' * (5 - int(round(cal)))
    print(f"{tienda}: {cal:.2f} / 5.00  {estrellas}")

print(f"\nTienda con MEJOR calificación: {calificacion.idxmax()} ({calificacion.max():.2f})")
print(f"Tienda con PEOR calificación:  {calificacion.idxmin()} ({calificacion.min():.2f})")

In [ ]:
# Gráfico circular - Distribución de calificaciones
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
tiendas_list = ['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']
colores_cal = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

for ax, nombre in zip(axes, tiendas_list):
    datos = df[df['Tienda'] == nombre]['Calificación'].value_counts().sort_index()
    wedges, texts, autotexts = ax.pie(
        datos, labels=[f'{i}⭐' for i in datos.index],
        autopct='%1.1f%%', colors=colores_cal, startangle=90,
        textprops={'fontsize': 9}
    )
    prom = df[df['Tienda'] == nombre]['Calificación'].mean()
    ax.set_title(f'{nombre}\nPromedio: {prom:.2f}', fontsize=11, fontweight='bold')

fig.suptitle('Distribución de Calificaciones por Tienda', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 4. 🏆 Productos Más y Menos Vendidos


In [ ]:
# Top 5 más vendidos y 5 menos vendidos por tienda
for nombre in ['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']:
    datos = df[df['Tienda'] == nombre]['Producto'].value_counts()
    print(f'\n=== {nombre} ===')
    print('  TOP 5 MÁS VENDIDOS:')
    for prod, cnt in datos.head(5).items():
        print(f'    {prod}: {cnt} ventas')
    print('  TOP 5 MENOS VENDIDOS:')
    for prod, cnt in datos.tail(5).items():
        print(f'    {prod}: {cnt} ventas')

In [ ]:
# Gráfico de barras horizontales - Top 5 productos más vendidos por tienda
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
tiendas_list = ['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']
colores_prod = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

for ax, nombre, color in zip(axes.flat, tiendas_list, colores_prod):
    top5 = df[df['Tienda'] == nombre]['Producto'].value_counts().head(5)
    bars = ax.barh(top5.index[::-1], top5.values[::-1], color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, top5.values[::-1]):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)
    ax.set_title(f'Top 5 productos - {nombre}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Número de Ventas')
    ax.xaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle('Productos Más Vendidos por Tienda', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 5. 🚚 Envío Promedio por Tienda


In [ ]:
# Costo de envío promedio por tienda
envio = df.groupby('Tienda')['Costo de envío'].mean().round(0).sort_values(ascending=False)

print('=== Costo de Envío Promedio por Tienda ===')
for tienda, costo in envio.items():
    print(f"{tienda}: ${costo:,.0f} COP")

print(f"\nTienda con envío MÁS CARO: {envio.idxmax()} (${envio.max():,.0f} COP)")
print(f"Tienda con envío MÁS BARATO: {envio.idxmin()} (${envio.min():,.0f} COP)")

In [ ]:
# Gráfico de dispersión - Precio promedio vs Costo de envío promedio
precio_prom = df.groupby('Tienda')['Precio'].mean()
envio_prom = df.groupby('Tienda')['Costo de envío'].mean()
ventas_total = df.groupby('Tienda').size()

fig, ax = plt.subplots(figsize=(8, 6))
colores_scatter = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

for i, tienda in enumerate(['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']):
    ax.scatter(precio_prom[tienda], envio_prom[tienda],
               s=ventas_total[tienda] / 3, color=colores_scatter[i],
               alpha=0.85, edgecolors='white', linewidth=1.5, label=tienda, zorder=3)
    ax.annotate(tienda, (precio_prom[tienda], envio_prom[tienda]),
                textcoords='offset points', xytext=(10, 5), fontsize=10, fontweight='bold')

ax.set_title('Precio Promedio vs Costo de Envío Promedio\n(tamaño = número de ventas)', fontsize=13, fontweight='bold')
ax.set_xlabel('Precio Promedio (COP)', fontsize=11)
ax.set_ylabel('Costo de Envío Promedio (COP)', fontsize=11)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# 6. 📊 Resumen Comparativo


In [ ]:
# Tabla resumen de todos los indicadores
resumen = pd.DataFrame({
    'Facturación Total': df.groupby('Tienda').apply(lambda x: (x['Precio'] + x['Costo de envío']).sum()),
    'N° Ventas': df.groupby('Tienda').size(),
    'Calificación Promedio': df.groupby('Tienda')['Calificación'].mean().round(2),
    'Envío Promedio': df.groupby('Tienda')['Costo de envío'].mean().round(0),
    'Precio Promedio': df.groupby('Tienda')['Precio'].mean().round(0),
})

print('=== TABLA RESUMEN ===')
print(resumen.to_string())

In [ ]:
# Gráfico radar / líneas comparativas normalizadas
from matplotlib.ticker import FuncFormatter

metricas = ['Facturación\nTotal', 'N° Ventas', 'Calificación\nPromedio', 'Envío\nPromedio', 'Precio\nPromedio']

# Normalizar entre 0 y 1 (min-max)
resumen_norm = (resumen - resumen.min()) / (resumen.max() - resumen.min())

fig, ax = plt.subplots(figsize=(10, 5))
colores_line = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

for i, tienda in enumerate(resumen_norm.index):
    ax.plot(metricas, resumen_norm.loc[tienda].values,
            marker='o', color=colores_line[i], linewidth=2.5,
            markersize=8, label=tienda)

ax.set_title('Comparación Normalizada de Indicadores por Tienda', fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Valor Normalizado (0=peor, 1=mejor)', fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
ax.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.show()

# 7. 📝 Recomendación Final

---

## ✅ Conclusión: El Sr. Juan debería vender la **Tienda 4**

Tras analizar los datos de las cuatro tiendas de Alura Store en los indicadores clave de facturación, calificaciones, volumen de ventas, productos y costos de envío, la evidencia apunta consistentemente a que **Tienda 4** es la menos eficiente y la candidata más razonable para ser vendida:

### 📉 Razones para vender Tienda 4:

1. **Menor facturación total:** La Tienda 4 registra los ingresos más bajos de la cadena, lo que indica menor capacidad de generar valor económico.

2. **Menor número de ventas:** Tiene el volumen de transacciones más reducido entre las cuatro tiendas, lo que refleja menor demanda o menor alcance comercial.

3. **Calificación promedio más baja:** Sus clientes le otorgan las peores puntuaciones en promedio, señal de insatisfacción y riesgo de pérdida de fidelidad.

4. **Costo de envío y precio promedio menores:** Aunque puede parecer una ventaja, en el contexto de bajo volumen se traduce en márgenes reducidos sin compensación por escala.

### 🏆 Tiendas a conservar:

- **Tienda 1** lidera en facturación y volumen de ventas → activo más valioso.
- **Tienda 2** y **Tienda 3** mantienen indicadores equilibrados y calificaciones competitivas.

### 💡 Recomendación:

> Vender la **Tienda 4** liberará capital para reinvertir en las tiendas con mejor desempeño, sin sacrificar ingresos significativos. Es la decisión estratégica más sólida basada en los datos.

---
*Análisis realizado con Python · Pandas · Matplotlib*
